ection 1: Business Problem
What are we solving?

Hospitals store data in multiple systems:

EHR System
Claims System
Laboratory System

Unfortunately:

Patient IDs differ
Diagnosis codes differ
Lab units differ

Before building:

Dashboards
AI Models
Population Health Reports

we must harmonize the data.

Real-World Example

Daniel Rivera appears in:

EHR
EHR-001
Claims
MEM001
Lab
LAB001

A computer thinks:

3 different patients

But reality:

1 patient

Our job is to fix this.

Section 2: Load Libraries
What does this code do?

Loads Pandas.

Pandas is the most common Python library for healthcare analytics.

Why do we use it?

It helps us:

Load data
Clean data
Merge data
Analyze data

Section 2: Load Libraries
What does this code do?

Loads Pandas.

Pandas is the most common Python library for healthcare analytics.

Why do we use it?

It helps us:

Load data
Clean data
Merge data
Analyze data


In [1]:
import pandas as pd

Section 3: Load Source Systems
What does this code do?

Loads three healthcare systems.

EHR

Contains:

Patient demographics
Encounters
Diagnoses
Claims

Contains:

Billing
Insurance
Diagnosis codes
Labs

Contains:

Lab test results
Units
Values

Why do we use it?

Real healthcare analytics never comes from one system.

In [2]:
ehr_df = pd.read_csv("/content/ehr_data.csv")

claims_df = pd.read_csv("/content/claims_data.csv")

labs_df = pd.read_csv("/content/labs_data.csv")

pd.read_csv()

Used whenever we load CSV data.

Section 4: Inspect Source Data
What does this code do?

Shows first few rows.

Why do we use it?

Before cleaning data, always inspect it.

In [3]:
ehr_df.head()

,ehr_patient_id,gender,age
0,P001,M,70
1,P002,F,41
2,P003,M,43
3,P004,M,77
4,P005,M,58


In [4]:
claims_df.head()

,claims_member_id,diagnosis_code,claim_amount
0,1,I10,23589
1,2,E11,48484
2,3,I10,22453
3,4,I10,41212
4,5,E11,48525


In [5]:
labs_df.head()

,lab_patient_id,test_name,result_value,unit
0,P-001,Glucose,5.1,mmol/L
1,P-002,Glucose,96.0,mmol/L
2,P-003,Glucose,74.0,mmol/L
3,P-004,Glucose,104.0,mmol/L
4,P-005,Glucose,101.0,mmol/L


df.head()

Always use after loading data.

Section 5: Investigate Patient IDs
What does this code do?

Looks at patient identifiers.

Why do we use it?

This is the most common healthcare integration problem.

Example:

EHR001
MEM001
LAB001

Same patient.

Different IDs.

In [6]:
print(ehr_df["ehr_patient_id"])
print(claims_df["claims_member_id"])
print(labs_df["lab_patient_id"])

0      P001
1      P002
2      P003
3      P004
4      P005
       ... 
115    P116
116    P117
117    P118
118    P119
119    P120
Name: ehr_patient_id, Length: 120, dtype: object
0        1
1        2
2        3
3        4
4        5
      ... 
115    116
116    117
117    118
118    119
119    120
Name: claims_member_id, Length: 120, dtype: int64
0      P-001
1      P-002
2      P-003
3      P-004
4      P-005
       ...  
115    P-116
116    P-117
117    P-118
118    P-119
119    P-120
Name: lab_patient_id, Length: 120, dtype: object


df["column"]

Use to inspect a specific colum

Section 6: Standardize Patient IDs
What does this code do?

Creates one common identifier.

Why do we use it?

Without this step:

Readmission Rate Wrong
Disease Prevalence Wrong
Patient Counts Wrong

In [8]:
ehr_df["patient_id"] = (
    ehr_df["ehr_patient_id"]
    .str.replace("P", "", regex=False) # Changed from EHR to P based on actual data like 'P001'
    .str.zfill(3)
)

claims_df["patient_id"] = (
    claims_df["claims_member_id"]
    .astype(str) # Convert to string to allow .str accessor
    .str.zfill(3) # Pad with leading zeros for consistent format (e.g., 1 becomes '001')
)

labs_df["patient_id"] = (
    labs_df["lab_patient_id"]
    .str.replace("P-", "", regex=False) # Changed from LAB to P- based on actual data like 'P-001'
    .str.zfill(3)
)

Section 7: Diagnosis Code Mapping
What does this code do?

Translates ICD codes into readable diagnoses.

Why do we use it?

Dashboards should show:

Type 2 Diabetes

not:

E11

In [9]:
diagnosis_map = {
    "I10": "Hypertension",
    "E11": "Type 2 Diabetes"
}

claims_df["diagnosis_name"] = (
    claims_df["diagnosis_code"]
    .map(diagnosis_map)
)

.map()

Used for terminology mapping.

Very common in healthcare.

ection 8: Standardize Lab Units
What does this code do?

Converts glucose measurements into a single unit.

Why do we use it?

One hospital may use:

mg/dL

Another:

mmol/L

Without conversion:

Average Glucose Wrong
Risk Scores Wrong

In [10]:
def convert_glucose_to_mgdl(value, unit):

    if unit == "mmol/L":
        return value * 18

    return value

In [13]:
labs_df["glucose_mgdl"] = labs_df.apply(
    lambda row: convert_glucose_to_mgdl(
        row["result_value"],
        row["unit"]
    ),
    axis=1
)

Section 9: Merge Systems
What does this code do?

Combines:

EHR
Claims
Labs

into one patient dataset.

Why do we use it?

This creates a 360° patient view.

In [14]:
merged_df = (
    ehr_df
    .merge(
        claims_df,
        on="patient_id",
        how="inner"
    )
    .merge(
        labs_df,
        on="patient_id",
        how="inner"
    )
)

df.merge()

Most important healthcare data engineering skill

Section 10: Validate Final Dataset
What does this code do?

Checks whether merge succeeded.

Why do we use it?

Never trust a merge until validated.

In [15]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ehr_patient_id    120 non-null    object 
 1   gender            120 non-null    object 
 2   age               120 non-null    int64  
 3   patient_id        120 non-null    object 
 4   claims_member_id  120 non-null    int64  
 5   diagnosis_code    120 non-null    object 
 6   claim_amount      120 non-null    int64  
 7   diagnosis_name    120 non-null    object 
 8   lab_patient_id    120 non-null    object 
 9   test_name         120 non-null    object 
 10  result_value      120 non-null    float64
 11  unit              120 non-null    object 
 12  glucose_mgdl      120 non-null    float64
dtypes: float64(2), int64(3), object(8)
memory usage: 12.3+ KB


In [16]:
merged_df.head()

,ehr_patient_id,gender,age,patient_id,claims_member_id,diagnosis_code,claim_amount,diagnosis_name,lab_patient_id,test_name,result_value,unit,glucose_mgdl
0,P001,M,70,001,1,I10,23589,Hypertension,P-001,Glucose,5.1,mmol/L,91.8
1,P002,F,41,002,2,E11,48484,Type 2 Diabetes,P-002,Glucose,96.0,mmol/L,1728.0
2,P003,M,43,003,3,I10,22453,Hypertension,P-003,Glucose,74.0,mmol/L,1332.0
3,P004,M,77,004,4,I10,41212,Hypertension,P-004,Glucose,104.0,mmol/L,1872.0
4,P005,M,58,005,5,E11,48525,Type 2 Diabetes,P-005,Glucose,101.0,mmol/L,1818.0


Section 11: Healthcare Data Quality Checks
What does this code do?

Checks for missing values.

Why do we use it?

Missing data breaks:

Dashboards
Quality Metrics
AI Models

In [17]:
merged_df.isnull().sum()

,0
ehr_patient_id,0
gender,0
age,0
patient_id,0
claims_member_id,0
diagnosis_code,0
claim_amount,0
diagnosis_name,0
lab_patient_id,0
test_name,0


In [ ]:
Section 12: Business Insight

Now we have harmonized healthcare data.

We can answer:

Clinical Questions
Which diabetic patients have abnormal glucose?
Operational Questions
How many patients had claims and lab results?
AI Questions
Who is at risk of hospitalization?